<a href="https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/sudoku/iLLaDA_Sudoku_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# iLLaDA Sudoku on Colab Pro

Run the `adi` branch with a persistent model copy in Google Drive. Select **Runtime → Change runtime type → GPU → L4 or A100** (subject to availability), then run the cells in order.

The first session downloads approximately 16 GB of weights into this model's own Drive folder. Later sessions reuse that pinned snapshot. Each new runtime still loads the weights into GPU memory. The optional copy to `/content` speeds repeated model loads within a session.

For the private GitHub repository, authorize Colab to read GitHub or upload this notebook. The clone cell separately needs a fine-grained token with **Contents: read** permission: store it in Colab **Secrets** as `GITHUB_TOKEN` or enter it into the hidden prompt. Tokens are never stored in the clone URL or notebook source.

The editable Config Updater below preserves your three-puzzle defaults. Use one puzzle for the first GPU verification. Evaluation is model-only: Sudoku rules validate answers, without repairing them.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or A100 GPU runtime, then reconnect.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | GPU memory: {gpu.total_memory / 2**30:.1f} GiB")
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Choose an L4/A100 runtime with more GPU memory; a 16 GB T4 is too small for this setup.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support the configured BF16 inference.")


## 1. Get the code from `adi`
Clone or update `adi`. A clean checkout on another branch can switch safely; local modifications and divergent commits are never discarded.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO_DIR = Path("/content/CSCI544-DLM")

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

ensure_checkout()


## 2. Install iLLaDA dependencies
Keep Colab's CUDA-enabled PyTorch. Inference always uses a fresh Python process, which sees the installed dependency versions. iLLaDA uses Transformers 4.57.1 and Tokenizers 0.22.2; the older LLaDA tokenizer dependencies cannot read this checkpoint.

If you switch model notebooks in the same runtime, run this installation cell again before inference. The two model profiles use different Transformers versions.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements/illada-colab.txt")], check=True)

# Install our package without replacing Colab's CUDA-enabled PyTorch.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_DIR)], check=True)
# Refresh editable-package paths in this already-running notebook kernel.
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)


## 3. Mount Drive
Allow at least **20 GB free Drive storage per model snapshot**, plus dataset cache and experiment outputs. Downloading both models needs room for both copies. Colab Pro compute and Drive storage are separate allowances.

`COPY_MODEL_TO_RUNTIME = True` stages the persistent model on runtime disk. Set it to `False` to load directly from Drive. Existing experiment folders are not moved or deleted.


In [ ]:
from google.colab import drive
import importlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM")
MODEL_FAMILY = "illada"
from dlm_project.models import profiles as model_profiles
from dlm_project.experiments import paths as experiment_paths
importlib.reload(model_profiles)
importlib.reload(experiment_paths)
profile = model_profiles.get_profile(MODEL_FAMILY)
MODEL_NAME = profile["repo_id"].split("/")[-1]
DRIVE_MODEL_DIR = DRIVE_ROOT / "models" / MODEL_NAME
RUNTIME_MODEL_DIR = Path("/content/models") / MODEL_NAME
COPY_MODEL_TO_RUNTIME = True
EXPERIMENT_TIMEZONE = "America/Los_Angeles"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
MODEL_PATH = RUNTIME_MODEL_DIR if COPY_MODEL_TO_RUNTIME else DRIVE_MODEL_DIR
print("Persistent model:", DRIVE_MODEL_DIR)
print("Experiment root:", DRIVE_ROOT / profile["label"] / "Sudoku")


## 4. Load this run's base configuration
Set the model, cache locations, and timezone. Choose sample count and generation parameters in the **Config Updater** below, which writes the final configuration and checks the data before inference.


In [ ]:
import yaml

config = yaml.safe_load((REPO_DIR / "configs/sudoku/illada.yaml").read_text())
config["data"]["cache_dir"] = os.environ["HF_DATASETS_CACHE"]
config["model"].update(family=MODEL_FAMILY, model_path=str(MODEL_PATH), device="cuda", local_files_only=True)
config["experiment_root"] = str(DRIVE_ROOT)
config["experiment_timezone"] = EXPERIMENT_TIMEZONE
RUN_CONFIG = Path("/content/illada-sudoku-run.yaml")
RUNNER = REPO_DIR / "scripts/run_illada_sudoku.py"
print("Model revision:", profile["revision"])


## 5. Cache the model on Drive
Download the pinned model's tokenizer, chat template, configuration, Python implementation, and all shards listed in its weight index. Later sessions reuse the same files; keep the model folder's hidden `.cache` directory.

Rerun this cell after an interrupted download. If validation reports missing files or insufficient disk space, resolve that before inference. No weights are committed to GitHub.


In [ ]:
command = [sys.executable, "-u", str(REPO_DIR / "scripts/cache_model.py"), "--model", MODEL_FAMILY, "--drive-dir", str(DRIVE_MODEL_DIR)]
if COPY_MODEL_TO_RUNTIME:
    command += ["--runtime-dir", str(RUNTIME_MODEL_DIR)]
subprocess.run(command, check=True)


## Config Updater
This is the editable configuration used by the next run. Defaults match your custom experiment. Set `NUM_SAMPLES = 1` for the first verification, then use `3` for the configured experiment.

The folder name follows the actual parameters. A timestamp in `DD:MM:YY:HH:mm:SS` format is appended by the runner, using the timezone chosen above. Same-second runs get a numeric suffix. The dataset's first preparation can process all one million rows even when you select one puzzle.


In [ ]:
NUM_SAMPLES = 3
config["data"]["num_samples"] = NUM_SAMPLES
config["generation"].update(
    gen_length=256,
    steps=256,
    block_length=128,
    temperature=0.0,
    remasking="low_confidence",
)

# Derive the experiment name from the exact configuration that will run.
exp_name = experiment_paths.experiment_name(config["generation"])
config["output_dir"] = str(experiment_paths.experiment_parent(config))
RUN_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print(f"Experiment output directory: {config['output_dir']}\n")
print(RUN_CONFIG.read_text())
subprocess.run([sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG), "--dataset-only"], check=True)


## 6. Run the experiment
The model loads once per invocation. Each completed prediction is saved as JSONL, and its plot is saved automatically to Drive. Metrics record how many predictions and plots have been saved. `status: completed` means all run artifacts were saved; it does not mean Sudoku was solved.

Each invocation creates a new directory. Interrupted runs retain completed work; automatic resume is not implemented. After changing settings, rerun the Config Updater before this cell.


In [ ]:
import json
import uuid

latest = None
# A unique receipt identifies this invocation; it cannot select an older run.
RUN_MANIFEST = Path("/content") / f"{MODEL_FAMILY}-run-{uuid.uuid4().hex}.json"
subprocess.run([
    sys.executable, "-u", str(RUNNER), "--config", str(RUN_CONFIG),
    "--run-manifest", str(RUN_MANIFEST),
], check=True)
receipt = json.loads(RUN_MANIFEST.read_text())
latest = Path(receipt["run_dir"])
print("This run:", latest)


## 7. Inspect all saved predictions and plots
All artifacts are in the current experiment's timestamp folder: `config.json`, `metrics.json`, `predictions.jsonl`, and `plots/puzzle_000.png`, etc.

The displayed images are the PNGs saved during evaluation. Incomplete answers keep their original row positions, complete invalid grids are flagged, and red digits conflict with given clues. Nothing is repaired by a solver.


In [ ]:
if latest is None:
    raise RuntimeError("Complete the run cell before inspecting its results.")
print("Results:", latest)
print(json.dumps(json.loads((latest / "metrics.json").read_text()), indent=2))
print("\n--- All Predictions ---")
with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        record = json.loads(line)
        print(f"\nPrediction for Puzzle Index {record['index']}:")
        print(json.dumps(record, indent=2))


In [ ]:
from IPython.display import Image, display

with (latest / "predictions.jsonl").open() as handle:
    for line in handle:
        current_record = json.loads(line)
        plot_path = latest / current_record["plot_file"]
        print(f"Puzzle Index {current_record['index']} | {plot_path}")
        display(Image(filename=str(plot_path)))


## 8. Verify caching, then increase the sample count
After the one-puzzle verification, set `NUM_SAMPLES = 3` in the Config Updater and rerun it and steps 6–7. To check reuse, rerun step 5 and evaluation: the pinned Drive snapshot and unchanged runtime files are reused, and a new experiment timestamp is created.

Review Sudoku validity and clue preservation before increasing to 10 or 100 puzzles. Change parameters only in the Config Updater; each run saves the resolved configuration. Old Drive result folders remain untouched.

### Notes and references
- Fixed-length masked generation uses the iLLaDA checkpoint's own tokenizer and chat template. There is no training, solver repair, or automatic variable-length decoding.
- Exact accuracy compares all 81 digits with the reference. Cell accuracy divides matching cells by `81 × processed puzzles`. Failed parsing produces an empty prediction and zero matching cells.
- Completion, clue-preservation, and Sudoku-validity rates use processed puzzles as their denominator. A complete grid may still be invalid.
- Config metadata records the model identity/revision, Git commit, run time, and timezone. Saved notebook outputs are cleared in Git; run the cells to obtain new results.
- [Model](https://huggingface.co/GSAI-ML/iLLaDA-8B-Instruct) · [Upstream sampler](https://github.com/ML-GSAI/LLaDA) · [Dataset](https://huggingface.co/datasets/certen/Sudoku-1M-Puzzles)
